In [2]:
1+1

2

In [13]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import  START, END
from langgraph.graph.state import  StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_core.tools import tool
from langchain_core.messages import BaseMessage
import os
from dotenv import load_dotenv
load_dotenv()

True

In [14]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGSMITH_TRACING"]= "true"



In [15]:
from langchain_groq import ChatGroq
# llm = init_chat_model('openai/gpt-oss-120b')

llm = ChatGroq( model = 'openai/gpt-oss-120b')
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x10ce7b250>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10ce7bc50>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [16]:
class State(TypedDict):
    messages:Annotated[list[BaseMessage], add_messages]

In [20]:
## Graph with tool call
from langchain_core.tools import tool

@tool
def add(a:float, b:float):
    """add two numbers"""
    return a+b

tools=[add]
tool_node = ToolNode([add])


llm_with_tool = llm.bind_tools([add])

def call_llm_model(state:State):
    return {"messages":[llm_with_tool.invoke(state["messages"])]}

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

#Node definition
def tool_calling_llm(state:State):
    return {"messages": [llm_with_tool.invoke(state["messages"])]}

#graph
builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tool", ToolNode(tools))

# add Edges
builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    #if the latest message (resust) from assistant is a tool call -> tools_condition routes to tools
    #if the latest message (result) from assistant is not a tool call -> tool_condition routes to END
    tools_condition
)
builder.add_edge("tool", END)

#compile graph
graph = builder.compile


from IPython.display import Image, display

# graph = builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

ValueError: At 'tool_calling_llm' node, 'tools_condition' branch found unknown target 'tools'